# Assignment 3: Logistic Regression

<font color=darkblue>
    
In this assignment you will perform logistic regression to classify the financial condition of a new bank.

Enter the code along with your comments in each question section.

## Import Libraries & Data Overview

The file **Banks.csv** includes data on a sample of 20 banks. 

The “Financial Condition” column records the judgment of an expert on the financial condition of each bank. This response variable takes one of two possible values—weak or strong—according to the financial condition of the bank. 

The predictors are two ratios used in the financial analysis of banks: TotLns&Lses/Assets is the ratio of total loans and leases to total assets and TotExp/Assets is the ratio of total expenses to total assets. 

The target is to classify the financial condition of a new bank using the two ratios.


### Q1.1 Load Packages & Import Dataset

In [3]:
# Load the required packages

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

#to scale the data using z-score 
from sklearn.preprocessing import StandardScaler

from sklearn.model_selection import train_test_split

#algorithms to use
from sklearn.linear_model import LogisticRegression

#Metrics to evaluate the model
from sklearn.metrics import confusion_matrix, classification_report, precision_recall_curve

#for tuning the model
from sklearn.model_selection import GridSearchCV

import statsmodels.formula.api as SM
#to ignore warnings
import warnings
warnings.filterwarnings("ignore")

In [47]:
# Import the dataset
from google.colab import files
uploaded = files.upload()


Saving Banks_2.csv to Banks_2.csv


## Logistic Regression Model

### Q2.1 Modeling

Run a logistic regression model (on the entire dataset) that models the status of a bank as a function of the two financial measures, TotLns&Lses/Assets and TotExp/Assets. 

Specify the success class as weak (this is similar to creating a dummy that is 1 for financially weak banks and 0 otherwise), and use the default cutoff value of 0.5.


In [54]:
data=pd.read_csv('Banks_2.csv')

#Changed column names from 
#TotCap/Assets, TotExp/Assets, TotLns&Lses/Assets to TotCapAssets, TotExpAssets, TotLnsLsesAssets

In [55]:
data.head

<bound method NDFrame.head of     Obs  FinancialCondition  TotCapAssets  TotExpAssets  TotLnsLsesAssets
0     1                   1           9.7          0.12              0.65
1     2                   1           1.0          0.11              0.62
2     3                   1           6.9          0.09              1.02
3     4                   1           5.8          0.10              0.67
4     5                   1           4.3          0.11              0.69
5     6                   1           9.1          0.13              0.74
6     7                   1          11.9          0.10              0.79
7     8                   1           8.1          0.13              0.63
8     9                   1           9.3          0.16              0.72
9    10                   1           1.1          0.16              0.57
10   11                   0          11.1          0.08              0.43
11   12                   0          20.5          0.12              0.80
12   13 

In [8]:
condition_weak = data.loc[(data.FinancialCondition == 1)]
condition_strong	= data.loc[(data.FinancialCondition == 0)]

In [62]:
X = data[['TotLnsLsesAssets', 'TotExpAssets']]
y = data[['FinancialCondition']]

In [63]:
f1= 'FinancialCondition ~ TotLnsLsesAssets + TotExpAssets'

model = SM.logit(formula=f1, data=data)
result = model.fit()
# summarize the model
print (result.summary())
#We now have the model and the summaries should provide the coefficient, intercept, standard errors and p-values

Optimization terminated successfully.
         Current function value: 0.328688
         Iterations 8
                           Logit Regression Results                           
Dep. Variable:     FinancialCondition   No. Observations:                   20
Model:                          Logit   Df Residuals:                       17
Method:                           MLE   Df Model:                            2
Date:                Wed, 08 Feb 2023   Pseudo R-squ.:                  0.5258
Time:                        03:25:32   Log-Likelihood:                -6.5738
converged:                       True   LL-Null:                       -13.863
Covariance Type:            nonrobust   LLR p-value:                 0.0006829
                       coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------
Intercept          -14.7210      6.675     -2.205      0.027     -27.804      -1.638
TotLnsLsesA

### Q2.2 Estimated Equations

Write the estimated equation that associates the financial condition of a bank with its two predictors in three formats:

a. The logit as a function of the predictors

b. The odds as a function of the predictors

c. The probability as a function of the predictors


In [68]:
#a)

# Get the parameters of the logistic regression model
intercept, coef1, coef2 = result.params

logit = intercept + coef1 * X["TotLnsLsesAssets"] + coef2 * X["TotExpAssets"]
logit

0     1.500420
1     0.350941
2     1.902791
3    -0.128832
4     0.936934
5     3.152178
6     0.875727
7     2.231333
8     5.679769
9     4.424071
10   -3.934627
11    2.756118
12   -2.656423
13   -3.097495
14   -0.524892
15   -4.124560
16   -3.432348
17   -0.235051
18   -3.850914
19   -3.264921
dtype: float64

In [69]:
#b)
odds = np.exp(logit)
odds


0       4.483572
1       1.420404
2       6.704581
3       0.879122
4       2.552144
5      23.386946
6       2.400619
7       9.312269
8     292.881793
9      83.435264
10      0.019553
11     15.738628
12      0.070199
13      0.045162
14      0.591619
15      0.016171
16      0.032311
17      0.790530
18      0.021260
19      0.038200
dtype: float64

In [70]:
#c)

prob = odds / (1 + odds)
prob

0     0.817637
1     0.586846
2     0.870207
3     0.467837
4     0.718480
5     0.958994
6     0.705936
7     0.903028
8     0.996597
9     0.988157
10    0.019178
11    0.940258
12    0.065594
13    0.043211
14    0.371709
15    0.015913
16    0.031300
17    0.441506
18    0.020818
19    0.036794
dtype: float64

## Classify Financial Condition of New Bank

### Q3.1 Estimates for New Bank

Consider a new bank whose total loans and leases/assets ratio = 0.6 and total expenses/assets ratio = 0.11. 

From your logistic regression model, estimate the following four quantities for this bank: 

the logit, the odds, the probability of being financially weak, and the classification of the bank (use cutoff = 0.5).

In [77]:
# a. The logit for the new bank
new_bank = pd.DataFrame({"TotLnsLsesAssets": [0.6], "TotExpAssets": [0.11]})
#new_bank = sm.add_constant(new_bank) # add an intercept column
new_bank

,TotLnsLsesAssets,TotExpAssets
0,0.6,0.11


In [79]:
logit_2 = result.predict(new_bank)
logit_2

0    0.54575
dtype: float64

In [81]:

# b. The odds for the new bank
odds_2 = np.exp(logit_2)
odds_2



0    1.725903
dtype: float64

In [83]:
# c. The probability of being financially weak for the new bank
prob_2 = odds_2 / (1 + odds_2)
prob_2

0    0.633149
dtype: float64

In [87]:
# d. The classification of the bank (use cutoff = 0.5)
classification_2 = np.where(prob_2 >= 0.5, 1, 0)
classification_2

array([1])

**Since the return value for classification_2 is 1 it means that the financial condition of this bank given a  leases/assets ratio = 0.6 and total expenses/assets ratio = 0.11 is more likely to be weak.**

### Q3.2 Classify New Bank

We use a cutoff value of 0.5 to classify a record based on propensity. 

Instead, if we want to classify the record using the odds or logit, what value should we take as a cutoff?

**This will depend on the cost for false positive or false negatiive predictions. In our case, the misclasification cost is higher when we clasify a bank as a strong bank but in reality it has a poor financial condition (weak). This indicates that if we were to use odds or logit, we would want to have have a low cutoff value such that it will decrease the chances of a weak bank being clasify as a strong bank.**

### Q3.3 Cutoff Value

When a bank with in poor financial condition is misclassified as financially strong, the misclassification cost is much higher than a financially strong bank misclassified as weak. 

To minimize the expected cost of misclassification, should the cutoff value for classification (which is currently at 0.5) be increased or decreased?

**We should decrease the cutoff value for classification. The reason is because if the probability of a bank being weak is greater than 0.5, we would classify it as financially weak. If it is less than 0.5, we classify it as strong. Therefore, by decreasing the cutoff value, the classification would be more rigorous where fewer banks that are actually in poor financial condition are not clasified as financially strong.** 